# Code-switched TTS with Qwen3-TTS — Mandarin ⇄ English

Qwen3-TTS is an LLM-based text-to-speech model, so it can speak **mixed-language text** in a single utterance. To generate *code-switched* speech you don't do anything exotic — you put **both languages in one `text` string** and let the model handle the switches, with **`language="auto"`**.

This notebook shows how to script that, using a **cloned voice**, with Mandarin↔English examples, and how to scale it to a manifest + SLURM.

## How to run
TTS needs a **GPU**. On this cluster GPU work must run under SLURM — start a kernel on a B300 node, e.g.:

```bash
srun -p gpu --gres=gpu:1 --cpus-per-task=8 --mem=64G --time=2:00:00 --pty bash
source ~/qwen3_tts/.venv/bin/activate
jupyter lab --no-browser --ip 0.0.0.0      # or point your IDE kernel at ~/qwen3_tts/.venv
```

Weights are cached on scratch (fetched by `python -m qwen3_tts_gen.download`).

## 1. Load the model

We use the raw `qwen_tts` API here so the technique is transferable. Two cluster-specific must-dos on the **B300 (Blackwell sm_103)**:

- `torch.backends.cuda.enable_cudnn_sdp(False)` — the cuDNN attention backend has no valid kernel on this GPU and **crashes inside the audio codec**; this routes SDPA to the working (mem-efficient/math) backends. *(The `qwen3_tts_gen` package does this for you inside `backend.load_model`.)*
- a **`cu128+`** PyTorch build (already in the project `.venv`).

In [1]:
import os, torch, numpy as np, soundfile as sf
from pathlib import Path
from IPython.display import Audio, display

# Shared weight cache on scratch (same one the CLI / SLURM jobs use).
os.environ.setdefault("QWEN3_TTS_CACHE", "/scratch/ick/qwen3_tts/cache")
os.environ.setdefault("HF_HOME", os.environ["QWEN3_TTS_CACHE"])

# B300 (sm_103): disable the broken cuDNN SDPA backend BEFORE generating.
torch.backends.cuda.enable_cudnn_sdp(False)

from qwen_tts import Qwen3TTSModel

MODEL_ID = "Qwen/Qwen3-TTS-12Hz-1.7B-Base"   # the voice-clone (Base) model
model = Qwen3TTSModel.from_pretrained(
    MODEL_ID, device_map="cuda:0", dtype=torch.bfloat16, attn_implementation="sdpa",
)
print("Supported languages:", model.get_supported_languages())

/home/ick/qwen3_tts/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/bin/sh: 1: sox: not found
SoX could not be found!

    If you do not have SoX, proceed here:
     - - - http://sox.sourceforge.net/ - - -

    If you do (or think that you should) have SoX, double-check your
    path variables.
    



********
********
 


Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 39756.44it/s]


Supported languages: ['auto', 'chinese', 'english', 'french', 'german', 'italian', 'japanese', 'korean', 'portuguese', 'russian', 'spanish']


## 2. The key idea: mixed text + `language="auto"`

`generate_voice_clone(text=, language=, ref_audio=, ref_text=)` synthesizes `text` in the voice of `ref_audio`. For code-switching:

- Put **both languages in `text`** — mixed scripts are fine: `今天的 meeting ...`.
- Set **`language="auto"`** so the model doesn't force a single language's phonology onto everything. (You can also pin the dominant language, e.g. `"Chinese"`; we compare below.)
- The **reference voice only controls timbre** — it's independent of the text language. A Mandarin or bilingual reference works well for zh↔en.

Supported values are `auto` + the **10** languages the model knows (Chinese, English, French, German, Italian, Japanese, Korean, Portuguese, Russian, Spanish). Code-switching works between **any pair of those** (English↔German too) — anything outside the 10 raises `Unsupported languages`.

In [ ]:
# Reference voice = QWEN-Adrian with the famous bee-nut line.
REF_AUDIO = "/home/ick/data/audio/qwenadrian.wav"
REF_TEXT  = "Four score and seven years ago I made a bee nut."

def synth(text, language="auto", ref_audio=REF_AUDIO, ref_text=REF_TEXT):
    """Voice-clone synthesis -> (mono float32 waveform, sample_rate)."""
    wavs, sr = model.generate_voice_clone(
        text=text, language=language, ref_audio=ref_audio, ref_text=ref_text,
    )
    return np.asarray(wavs[0]).squeeze().astype("float32"), sr

## 3. Generate code-switched examples

Each line mixes Mandarin and English. We save a WAV and play it inline.

In [3]:
examples = {
    "zh_dominant": "我最近在学习 machine learning，这个 framework 真的非常 powerful。",
    "en_dominant": "I really love Chinese food, especially 麻辣火锅 and 小笼包.",
    "mixed_work":  "今天的 meeting 我们先 review 一下 Q3 的 KPI，然后再讨论下个季度的 roadmap。",
    "names_terms": "请把这个 bug 分配给 Alice，她对 backend 的 API 最熟悉。",
    "bee-nut" : "四分，七年前我获得了 bee-nut"
}

OUT = Path("/scratch/ick/qwen3_tts/codeswitch"); OUT.mkdir(parents=True, exist_ok=True)
for name, text in examples.items():
    wav, sr = synth(text, language="auto")
    sf.write(str(OUT / f"{name}.wav"), wav, sr)
    print(f"[{name}] {len(wav)/sr:.1f}s  |  {text}")
    display(Audio(wav, rate=sr))

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


[zh_dominant] 5.0s  |  我最近在学习 machine learning，这个 framework 真的非常 powerful。


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


[en_dominant] 3.8s  |  I really love Chinese food, especially 麻辣火锅 and 小笼包.


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


[mixed_work] 6.7s  |  今天的 meeting 我们先 review 一下 Q3 的 KPI，然后再讨论下个季度的 roadmap。


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


[names_terms] 4.8s  |  请把这个 bug 分配给 Alice，她对 backend 的 API 最熟悉。


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


[bee-nut] 3.1s  |  四分，七年前我获得了 bee-nut


## 4. `language="auto"` vs pinning a language

Same code-switched sentence, three settings. Listen for which keeps the English words natural while pronouncing the Mandarin correctly. `auto` is usually best for heavy mixing; pinning the dominant language can help if `auto` mis-detects on very short inputs.

In [6]:
text = examples["bee-nut"]
for lang in ["auto", "Chinese", "English"]:
    wav, sr = synth(text, language=lang)
    print(f"language={lang!r}: {len(wav)/sr:.1f}s")
    display(Audio(wav, rate=sr))

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


language='auto': 655.3s


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


language='Chinese': 3.0s


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


language='English': 3.0s


## 5. Scaling up: code-switched manifests + SLURM

To generate many code-switched clips, write them as a JSONL manifest (one row per line) and use the project's batch CLI / SLURM array — same as any other manifest, just with mixed `text` and `language: "auto"`.

In [5]:
import json
rows = [
    {"id": f"cs_{i:03d}", "mode": "clone",
     "ref_audio": REF_AUDIO, "ref_text": REF_TEXT,
     "text": t, "language": "auto"}
    for i, t in enumerate(examples.values())
]
mpath = Path("/home/ick/qwen3_tts/manifests/codeswitch_demo.jsonl")
mpath.write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in rows) + "\n", encoding="utf-8")
print("wrote", mpath, f"({len(rows)} rows)")

wrote /home/ick/qwen3_tts/manifests/codeswitch_demo.jsonl (5 rows)


Then generate from the shell on a GPU node:

```bash
cd ~/qwen3_tts
# single job:
slurm/submit.sh generate manifests/codeswitch_demo.jsonl --out-dir /scratch/ick/qwen3_tts/codeswitch_batch
# or shard a big manifest across GPUs (resumable):
slurm/submit.sh array 8 manifests/codeswitch_big.jsonl --out-dir /scratch/ick/qwen3_tts/codeswitch_batch
```

### Tips
- **`language="auto"` is the enabler** for mixed text; without it the model applies one language's phonology to the whole string.
- The **reference voice sets timbre only** — pick a clean clip; a bilingual/Mandarin reference tends to give the most natural zh↔en switches.
- Keep **punctuation** (`，。？`) — it helps prosody and phrase boundaries at switch points.
- Works for **any pair of the 10 supported languages**, not just zh/en.
- Quality is best judged **by ear** — generate a few variants and listen.